# BERT-based methods

# Name: Andrew Yu


# Overview

In this assignment, you will be guided to develop and evaluate BioNLP methods for biomedical named entity recognition (BioNER). BioNER aims to identify entities of interest from biomedical text. In this homework, you will focus on BERT-based fine-tuning approaches and optionally explore LLM-based prompting approaches for recognizing chemical entities in PubMed abstracts.

This assignment is designed to provide a practical understanding of when different modeling approaches are appropriate for information extraction tasks.

## Name Entity Recognition

**Name Entity Rocognition** is a classic task in natural language processing. It means to locate and classify name entities in text into pre-defined categories, such as person names, locations, organizations. For us, our goal is to locate the chemical entity from *PubMed* abstracts.

Differing with many classification tasks, NER requires the model to output a label for every token. For example, in our task, if the model encounters the following sentence:

> Selegiline-induced postural hypotension in Parkinson's disease: a longitudinal study on the effects of drug withdrawal.

It will be tokenized as the following:

> `['Selegiline', '-', 'induced', 'postural', 'hypotension', 'in', 'Parkinson', ''', 's', 'disease', ':', 'a', 'longitudinal', 'study', 'on', 'the', 'effects', 'of', 'drug', 'withdrawal', '.']`

And the model is expected to output:

> `[B, I, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]`

With
- `B` marks the beginning of a chemical entity
- `I` means the inside of a chemical entity
- `O` represents outside a chemical entity

***

# Question 1

What's the advantage of using the `BIO` format as a label, instead of simply framing the label as `Yes` (the token is a chemical entity) or `No` (the token is NOT a chemical entity) ?

Hint: Think about when two chemical entities appear consecutively.

**Your Answer Here** <br>
The advantage of using the BIO format is that it allows for boundaries to be defined. For example, when two chemical entities appear consecutively, BIO allows you to tell if it's two separate entities (BB) or a single entity (BI).

***

## Fine Tuning BERT on NER

Before you start this assignment, you need to know the overall pipeline and the role which pre-trained BERT model played during this task. BERT is not originally trained for the NER task. It is made up with a massive stacks of transformer layers, an Masked Language Modeling head (predict the masked word), and a Next Sentence Prediction head (output true or false). These two heads are designed specifically for BERT's original training task, but are useless in our scenario. However, those transformer layers' jobs are purely feature extraction, which is trained on massive contexts to turn raw text into the rich, contextualized vectors. These high dimentional vectors are thought to be able to capture a high-level relations among words withinin their contexts.

In this homework, we will only be using those pretrained transformer layers of BERT. Instead of keeping its original heads, we will add our own classification head that is suitable for NER task. We can say we are fine-tuning the BERT model to learn the specific task.

## 1.1 Dependencies and Dataset
You are provided with a widely used BioNER dataset for identifying chemical entities from PubMed abstracts. The dataset has been processed into two formats: one for BERT-based models and one for LLM-based prompting.

Referring the dataset and the format of the label of BERT from [previous section](#scrollTo=I8_w4CvKh2J5) and the format of the LLM [Here]()

<h3 align="center">We will first guide you through the BERT approach.</h3>

Start by installing the required Python packages. Then upload `datasets.zip`, which contains the train/dev/test files in BIO format.

In [ ]:
!pip install transformers datasets seqeval evaluate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


Next, upload `datasets.zip` and unzip it. This gives us three splits:
- `train.tsv`: used to fit model parameters
- `dev.tsv`: used to evaluate and tune settings
- `test.tsv`: used to generate final submission predictions

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving datasets.zip to datasets.zip


In [ ]:
!unzip datasets.zip

Archive:  datasets.zip
 extracting: datasets/BERT_format/dev.tsv  
 extracting: datasets/BERT_format/test.tsv  
 extracting: datasets/BERT_format/train.tsv  
 extracting: datasets/LLM_format/sentence_level_dev.csv  
 extracting: datasets/LLM_format/sentence_level_test.csv  
 extracting: datasets/LLM_format/sentence_level_train.csv  


In [ ]:
# Manually check if we have all files we need
!ls datasets/BERT_format

dev.tsv  test.tsv  train.tsv


In [ ]:
# Proceed only if you have passed the following test
import os

expected_files = {"dev.tsv", "test.tsv", "train.tsv"}
actual_files = set(os.listdir("datasets/BERT_format"))

assert expected_files.issubset(actual_files), f"Missing files: {expected_files - actual_files}"

## 1.2 Read the BIO-formatted NER files

Let's check what the data look like first.

In [ ]:
def peek(file_path, lines=5):
    with open(file_path, "r", encoding="utf-8") as f:
        for i in range(lines):
            line = f.readline()
            print(repr(line))

peek("datasets/BERT_format/train.tsv")

'Selegiline\tB\n'
'-\tO\n'
'induced\tO\n'
'postural\tO\n'
'hypotension\tO\n'


***

# Question 2

Briefly describe how the BIO data format is framed.

Hint: Adjust the lines argument of the `peek` helper to observe how different sentences are separated.

**Your Answer Here** <br>
The BIO data is formatted where each line has one token and one label, and sentences are separated by blank lines.

***

Knowing the data format, we need to parse the data files into Python lists.

After this step, each sentence is represented as:
- a list of tokens (words)
- a list of labels (`B`, `I`, `O`) for train/dev only

In [ ]:
def read_ner_file(file_path, has_labels=True):
    sentences = []
    labels = [] if has_labels else None

    current_tokens = []
    current_labels = [] if has_labels else None

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()

            # Sentence boundary
            if line == "":
                if current_tokens:
                    sentences.append(current_tokens)
                    if has_labels:
                        labels.append(current_labels)
                    current_tokens = []
                    if has_labels:
                        current_labels = []
                continue

            parts = line.split("\t")

            if has_labels:
                if len(parts) < 2:
                    continue
                token = parts[0]
                label = parts[1]
                current_tokens.append(token)
                current_labels.append(label)
            else:
                token = parts[0]
                current_tokens.append(token)

    if current_tokens:
        sentences.append(current_tokens)
        if has_labels:
            labels.append(current_labels)

    return sentences, labels

With the defined parser `read_ner_file`, let's parse our data and check the results' format.

In [ ]:
train_sentences, train_labels = read_ner_file("datasets/BERT_format/train.tsv")
dev_sentences, dev_labels = read_ner_file("datasets/BERT_format/dev.tsv")
test_sentences, _ = read_ner_file("datasets/BERT_format/test.tsv", has_labels=False)

In [ ]:
print(f"The first sentence in the train dataset is:\n{" ".join(train_sentences[0])}\n")
print(f"It has labels of:\n{' '.join(train_labels[0])}\n")

The first sentence in the train dataset is:
Selegiline - induced postural hypotension in Parkinson ' s disease : a longitudinal study on the effects of drug withdrawal .

It has labels of:
B O O O O O O O O O O O O O O O O O O O O



In [ ]:
# Proceed only if you have passed the following tests

assert (len(train_sentences), len(dev_sentences), len(test_sentences)) == (4560, 4581, 4797)

Remember that neural networks can only understand numbers, not label strings. So we need to create two mapping tables to tokenize our input data:
- `label2id`: converts label names (e.g., `B`) to integers
- `id2label`: converts model outputs back to readable labels

These mappings are required for both training and evaluation.

In [ ]:
all_train_labels = [label for sentence_labels in train_labels for label in sentence_labels]
label_list = sorted(list(set(all_train_labels)))

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

for label in label_list:
    print(f"The mapping will map {label} to {label2id[label]}")

The mapping will map B to 0
The mapping will map I to 1
The mapping will map O to 2


## 1.3 Convert the data into Hugging Face Dataset objects

We wrap the token/label lists into Hugging Face `Dataset` objects. This gives us a standard data interface that works directly with tokenizer preprocessing, model training (`Trainer`), and evaluation.

Refer to the document [here](https://huggingface.co/docs/datasets/v4.8.4/en/package_reference/main_classes#datasets.Dataset.from_dict) to learn how to use the method `from_dict` correctly.

***

# TODO 1

In [ ]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_dict(
    # TODO: Build the training split dictionary.
    # Hint: include tokens and NER labels.
    {
        "tokens": train_sentences,
        "ner_tags": train_labels
    }
)

dev_dataset = Dataset.from_dict(
    # TODO: Build the validation split dictionary.
    # Hint: include tokens and NER labels.
    {
        "tokens": dev_sentences,
        "ner_tags": dev_labels
    }
)

test_dataset = Dataset.from_dict(
    # TODO: Build the test split dictionary.
    # Hint: test split only has tokens.
    {
        "tokens": test_sentences
    }
)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": dev_dataset,
    "test": test_dataset
})

In [ ]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 4560
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 4581
    })
    test: Dataset({
        features: ['tokens'],
        num_rows: 4797
    })
})

In [ ]:
# Proceed only if you have passed the following tests
assert set(dataset_dict.keys()) == {"train", "validation", "test"}, f"Expected splits {expected_splits}, but got {actual_splits}"
assert dataset_dict["train"].num_rows == 4560, f"Train rows mismatch: {dataset_dict['train'].num_rows}"
assert dataset_dict["validation"].num_rows == 4581, f"Validation rows mismatch: {dataset_dict['validation'].num_rows}"
assert dataset_dict["test"].num_rows == 4797, f"Test rows mismatch: {dataset_dict['test'].num_rows}"
assert dataset_dict["train"].column_names == ["tokens", "ner_tags"]
assert dataset_dict["validation"].column_names == ["tokens", "ner_tags"]
assert dataset_dict["test"].column_names == ["tokens"]

***

## 1.4 Tokenize the input and align the labels

BERT uses subword tokenization, so one original word may be split into multiple pieces.

For example, in BERT, the input `["Selegiline", "is", "good"]` will be tokenized to `[[CLS], Se, ##legi, ##line, is, good, [SEP]]` with 7 tokens in total, where `[CLS]` marks the start of a sentence and `[SEP]` marks the end. But in NER task, we only have 3 labels `[B, O, O]`. Since our labels are defined at the word level, so we must align them with tokenized output.

Rule used here:
- assign the original label `0, 1, 2` using `label2id` to the first subword piece
- assign `-100` to special tokens and remaining subword pieces so they are ignored in loss computation
- assign `0` as a place holder to the test set without `ner_tags`

Under this rule, the results from subword tokenization will be re-labeled as:
- `[-100, 0, -100, -100, 1, 2, -100]` in train/validation set
- `[-100, 0, -100, -100, 0, 0, -100]` in test set

***

# TODO 2

In [ ]:
from transformers import AutoTokenizer

def tokenize_and_align_labels(examples):
    # Later, we will be applied BERT subword tokenization to our input
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        # this tells the tokenizer the input is a list of
        # tokens separated by space
        is_split_into_words=True
    )

    aligned_labels = []

    # train/validation have ner_tags; test split does not.
    has_labels = "ner_tags" in examples

    for i in range(len(examples["tokens"])):
        # word_ids will tell you the map of the current subword token to
        # the index of the original NER token
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        if has_labels:
            labels = examples["ner_tags"][i]

        #################################################################
        # TODO: Replace the following blocks to the correct code
        # (~1 line) for each TODO
        for word_idx in word_ids:
            if word_idx is None:
                #label_ids.append("#TODO: Special tokens (e.g. [CLS]) should be ignored.")
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                #label_ids.append("#TODO: First subword of a word gets the word label.")
                if has_labels:
                    label_ids.append(label2id[labels[word_idx]])
                else:
                    label_ids.append(0) # Placeholder for test set without ner_tags
            else:
                #label_ids.append("#TODO: Non-first subwords (e.g. ##legi in Selegiline) should be ignored.")
                label_ids.append(-100)
            previous_word_idx = word_idx
        #################################################################

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

***

Then, we will use our helper to correct the labeling after applying BERT's subword tokenization.

In [ ]:
model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenized_datasets = dataset_dict.map(
    tokenize_and_align_labels,  # applied our helper here
    batched=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/4560 [00:00<?, ? examples/s]

Map:   0%|          | 0/4581 [00:00<?, ? examples/s]

Map:   0%|          | 0/4797 [00:00<?, ? examples/s]

Check the following output to see if it makes sense to you. Refer the answer format [here](#scrollTo=TJ2CU-Pe1Bbj)

In [ ]:
def peek_aligned(dataset="train", idx=0):
    assert dataset in ["train", "validation", "test"], f"Invalid dataset: {dataset}"
    assert idx in range(len(dataset_dict[dataset])), f"Invalid index: {idx}"

    sentence = tokenized_datasets[dataset][idx]
    original_tokens = sentence["tokens"]
    original_labels = sentence["ner_tags"] if "ner_tags" in sentence else None
    input_ids = sentence["input_ids"]
    aligned_labels = sentence["labels"]
    subwords = tokenizer.convert_ids_to_tokens(input_ids)

    print("Original Tokens       :  ", original_tokens)
    print("Original NER Tags     :  ", original_labels)
    print("-" * 80)
    print("After BERT's Subword  :  ", subwords)
    print("Aligned with Original :  ", aligned_labels)

In [ ]:
peek_aligned("train", 0)

Original Tokens       :   ['Selegiline', '-', 'induced', 'postural', 'hypotension', 'in', 'Parkinson', "'", 's', 'disease', ':', 'a', 'longitudinal', 'study', 'on', 'the', 'effects', 'of', 'drug', 'withdrawal', '.']
Original NER Tags     :   ['B', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
--------------------------------------------------------------------------------
After BERT's Subword  :   ['[CLS]', 'Se', '##leg', '##ili', '##ne', '-', 'induced', 'post', '##ural', 'h', '##y', '##pot', '##ens', '##ion', 'in', 'Parkinson', "'", 's', 'disease', ':', 'a', 'longitudinal', 'study', 'on', 'the', 'effects', 'of', 'drug', 'withdrawal', '.', '[SEP]']
Aligned with Original :   [-100, 0, -100, -100, -100, 2, 2, 2, -100, 2, -100, -100, -100, -100, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, -100]


In [ ]:
peek_aligned("test", 21)

Original Tokens       :   ['The', 'effects', 'of', 'quinine', 'and', '4', '-', 'aminopyridine', 'on', 'conditioned', 'place', 'preference', 'and', 'changes', 'in', 'motor', 'activity', 'induced', 'by', 'morphine', 'in', 'rats', '.']
Original NER Tags     :   None
--------------------------------------------------------------------------------
After BERT's Subword  :   ['[CLS]', 'The', 'effects', 'of', 'q', '##uin', '##ine', 'and', '4', '-', 'amino', '##py', '##rid', '##ine', 'on', 'conditioned', 'place', 'preference', 'and', 'changes', 'in', 'motor', 'activity', 'induced', 'by', 'm', '##or', '##phine', 'in', 'rats', '.', '[SEP]']
Aligned with Original :   [-100, 0, 0, 0, 0, -100, -100, 0, 0, 0, 0, -100, -100, -100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100, -100, 0, 0, 0, -100]


****

# Question 3

Does the output make sense to you? Explain why.

**Your Answer Here** <br>
The output does make sense to me. When looking at one of the train sentences, we can see that the first subword has one of the three BIO values while everything else has -100. For the test sentence example, the only values are 0 and -100, which makes sense since the test data does not have any ner_tags.

****

## 1.5 Load the token classification model

We load a pretrained `bert-base-cased` model and attach a token-classification head. You can think of this as: BERT provides contextual word representations, and the new head predicts one NER tag per token as we have explained in [previous section](#scrollTo=0HTvLrsFh2J6).

For exploration, you are encouraged to try other existing fine-tuned BERT-family models and compare their validation performance. You can feel free to use any BERT-based model to generate the final prediction file.

***

# TODO 3

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    #################################################################
    # TODO: Fill in required arguments.
    # Hint: pass checkpoint + label config (num_labels, id2label, label2id).
    # You should be quite familiar with this (HW1 and Midterm), maybe.
    # (~4 lines)
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
    #################################################################
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

***

## 1.6 Evaluation metric

We evaluate NER performance with `seqeval`, a standard library for sequence labeling tasks.

Reported metrics:
- Precision: how many predicted entities are correct
- Recall: how many gold entities are found
- F1: harmonic mean of precision and recall (main model-selection metric here)
- Accuracy: token-level label accuracy

***

# Question 4

Explains each metric's job. In other words, what do they measure? Why can't we just rely on a single metric (explains the potential consequence)? <br>
Precision: TP / (TP + FP). Measures how often a positive prediction is correct. <br>
Recall: TP / (TP + FN). Measures how many actual positives are predicted correctly. <br>
F1: 2((Precision x Recall) / (Precision + Recall)). Harmonic mean of precision and recall. Will be used as the main metric for model selection. <br>
Accuracy: (TP + TN) / (TP + TN + FP + FN). Measures the amount of correct predictions. <br>
We cannot just rely on a single metric because the different metrics measure differet ways a model can perform. For example, having high precision but low recall would mean that whenever the model predicts true, it's very likely that it's correct, but it can be predicting a lot of other positives as false. And looking at it backwards, high recall and low precision would mean that the model predicts true for nearly all positive values, but that can be done through guessing true most of the time and getting a lot of false positives.

****

***

# TODO 4

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    #################################################################
    # TODO: Convert logits to predicted label IDs.
    # The prediction should be the class with the
    # largest logits among all the others
    # (~1 line)
    #predictions = "YOUR ANSWER HERE"
    predictions = np.argmax(logits, axis=-1)
    #################################################################

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        current_predictions = []
        current_labels = []

        for pred_id, label_id in zip(prediction, label):
            #################################################################
            # TODO: skip where label_id is -100.
            # (~1-2 lines)
            #"YOUR ANSWER HERE"
            #"YOUR ANSWER HERE"
            if label_id == -100:
                continue
            #################################################################
            current_predictions.append(id2label[pred_id])
            current_labels.append(id2label[label_id])

        true_predictions.append(current_predictions)
        true_labels.append(current_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

***

Next, we create a data collator. It dynamically pads each batch to the needed length at runtime, so we do not have to pad every sentence to a global maximum length in advance.

This makes batching simpler and usually more memory-efficient.

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## 1.7 Training arguments

Here we define key hyperparameters and training behavior, including learning rate, batch size, number of epochs, when to evaluate/save checkpoints, and how to choose the best checkpoint.

For homework experiments, this is the main section you will tune. Refer to the [Hugging Face documentation](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.TrainingArguments) for all the arguments.

***

# TODO 5

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    #################################################################
    # TODO: Fill in all training arguments.
    # Suggested keys: output_dir, eval_strategy, save_strategy, learning_rate,
    # per_device_train_batch_size, per_device_eval_batch_size, num_train_epochs,
    # weight_decay, logging_steps, load_best_model_at_end,
    # metric_for_best_model, greater_is_better, save_total_limit.
    output_dir="my_model",
    eval_strategy="epoch",
    save_strategy = "epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2
    #################################################################
)

****

## 1.8 Train the model

We now initialize Hugging Face `Trainer` and fine-tune BERT on the training split. At the end of each epoch, the model is evaluated on the development split, and the best checkpoint is kept according to dev F1.

**Practical note**: Please run training on a GPU. CPU training can be very slow. In Google Colab, enable GPU via `Runtime -> Change runtime type -> T4 GPU` (or another available GPU).

***

# TODO 6

In [ ]:
from transformers import Trainer

trainer = Trainer(
    #################################################################
    # TODO: Fill in required Trainer arguments.
    # Hint: include model, training args, train/eval datasets,
    # tokenizer (processing_class), data_collator, and compute_metrics.
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
    #################################################################
)

***

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.000338,0.089094,0.931683,0.879933,0.905069,0.988540
2,0.004018,0.077898,0.902760,0.911539,0.907128,0.988668
3,0.002409,0.066704,0.908361,0.910230,0.909295,0.989068
4,0.000377,0.072047,0.912238,0.911726,0.911982,0.989417
5,0.000257,0.074851,0.906839,0.917524,0.912150,0.989289


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1425, training_loss=0.0012622647064359114, metrics={'train_runtime': 625.0403, 'train_samples_per_second': 36.478, 'train_steps_per_second': 2.28, 'total_flos': 971061016967712.0, 'train_loss': 0.0012622647064359114, 'epoch': 5.0})

## 1.9 Evaluation on development set

We have provided you an evaluation script (`ner_eval.py`) that compares BIO predictions against gold labels and reports precision/recall/F1/accuracy.

This script is useful for reproducible model comparison when you try different hyperparameters.

In [ ]:
%%writefile ner_eval.py
import argparse
import sys
from pathlib import Path
import evaluate

def read_gold_file(file_path):
    sentences = []
    labels = []
    current_tokens = []
    current_labels = []

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.strip() == "":
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_labels)
                    current_tokens = []
                    current_labels = []
                continue

            parts = line.split("\t")
            if len(parts) < 2:
                raise ValueError(f"Malformed line in gold file: {repr(line)}")

            current_tokens.append(parts[0])
            current_labels.append(parts[1])

    if current_tokens:
        sentences.append(current_tokens)
        labels.append(current_labels)

    return sentences, labels

def read_prediction_file(file_path):
    sentences = []
    labels = []
    current_tokens = []
    current_labels = []

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.strip() == "":
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_labels)
                    current_tokens = []
                    current_labels = []
                continue

            parts = line.split("\t")
            if len(parts) < 2:
                raise ValueError(f"Malformed line in prediction file: {repr(line)}")

            current_tokens.append(parts[0])
            current_labels.append(parts[1])

    if current_tokens:
        sentences.append(current_tokens)
        labels.append(current_labels)

    return sentences, labels

def validate_alignment(gold_sentences, pred_sentences, gold_labels, pred_labels):
    if len(gold_sentences) != len(pred_sentences):
        raise ValueError(
            f"Sentence count mismatch: gold has {len(gold_sentences)} sentences, "
            f"but prediction has {len(pred_sentences)} sentences."
        )

    for sent_idx, (gold_tokens, pred_tokens, gold_seq, pred_seq) in enumerate(
        zip(gold_sentences, pred_sentences, gold_labels, pred_labels)
    ):
        if len(gold_tokens) != len(pred_tokens):
            raise ValueError(
                f"Token count mismatch in sentence {sent_idx}: "
                f"gold has {len(gold_tokens)} tokens, prediction has {len(pred_tokens)} tokens."
            )

        if len(gold_seq) != len(pred_seq):
            raise ValueError(
                f"Label count mismatch in sentence {sent_idx}: "
                f"gold has {len(gold_seq)} labels, prediction has {len(pred_seq)} labels."
            )

        for tok_idx, (gold_token, pred_token) in enumerate(zip(gold_tokens, pred_tokens)):
            if gold_token != pred_token:
                raise ValueError(
                    f"Token mismatch at sentence {sent_idx}, token {tok_idx}: "
                    f"gold token is {repr(gold_token)}, prediction token is {repr(pred_token)}."
                )

def compute_scores(gold_labels, pred_labels):
    seqeval = evaluate.load("seqeval")
    results = seqeval.compute(predictions=pred_labels, references=gold_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

def main():
    parser = argparse.ArgumentParser(description="Evaluate BIO-formatted NER predictions.")
    parser.add_argument("--gold", type=str, required=True)
    parser.add_argument("--pred", type=str, required=True)
    args = parser.parse_args()

    gold_sentences, gold_labels = read_gold_file(args.gold)
    pred_sentences, pred_labels = read_prediction_file(args.pred)

    validate_alignment(gold_sentences, pred_sentences, gold_labels, pred_labels)
    scores = compute_scores(gold_labels, pred_labels)

    print("Evaluation Results")
    print("------------------")
    print(f"Precision: {scores['precision']:.4f}")
    print(f"Recall:    {scores['recall']:.4f}")
    print(f"F1:        {scores['f1']:.4f}")
    print(f"Accuracy:  {scores['accuracy']:.4f}")

if __name__ == "__main__":
    main()

Overwriting ner_eval.py


Next, run inference on the validation set. The model outputs label IDs, so we convert them back to label names (`B`, `I`, `O`) for readability and evaluation.

In [ ]:
predictions, labels, metrics = trainer.predict(tokenized_datasets["validation"])
predicted_ids = np.argmax(predictions, axis=-1)

val_pred_labels = []

for prediction, label in zip(predicted_ids, labels):
    current_predictions = []
    for pred_id, label_id in zip(prediction, label):
        if label_id == -100:
            continue
        #current_predictions.append(TODO)
        current_predictions.append(id2label[pred_id])
    val_pred_labels.append(current_predictions)

print(dev_sentences[0])
print(val_pred_labels[0])

['22', '-', 'oxacalcitriol', 'suppresses', 'secondary', 'hyperparathyroidism', 'without', 'inducing', 'low', 'bone', 'turnover', 'in', 'dogs', 'with', 'renal', 'failure', '.']
['B', 'I', 'I', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


Save validation predictions to a TSV file. Keeping this file makes it easy to inspect model errors and run the external evaluation script.

In [ ]:
def save_predictions(output_path, sentences, pred_labels):
    with open(output_path, "w", encoding="utf-8") as f:
        for tokens, preds in zip(sentences, pred_labels):
            for token, pred in zip(tokens, preds):
                f.write(f"{token}\t{pred}\n")
            f.write("\n")

save_predictions(
    "validation_prediction_6.tsv",
    dev_sentences,
    val_pred_labels
)

Run evaluation on the validation predictions and use the scores for model selection and hyperparameter tuning.

If performance is not satisfactory, go back to [Section 1.7](#scrollTo=VkcqQtLPUWH1), adjust hyperparameters, retrain, and compare validation F1 again.

In [ ]:
!python ner_eval.py --gold datasets/BERT_format/dev.tsv --pred validation_prediction_6.tsv

Evaluation Results
------------------
Precision: 0.9068
Recall:    0.9175
F1:        0.9122
Accuracy:  0.9893


Strategy: adjust learning_rate (1e-5, 2e-5, 3e-5) and num_train_epochs (2, 3, 5)<br>
Evaluation Results <br>
2e-5, 3 (validation_prediction.tsv):
Precision: 0.8947
Recall:    0.9121
F1:        0.9033
Accuracy:  0.9891 <br>
1e-5, 3 (validation_prediction_2.tsv):
Precision: 0.8978
Recall:    0.9136
F1:        0.9056
Accuracy:  0.9889 <br>
3e-5, 3 (validation_prediction_3.tsv):
Precision: 0.9024
Recall:    0.9108
F1:        0.9066
Accuracy:  0.9892 <br>
2e-5, 2 (validation_prediction_4.tsv):
Precision: 0.9013
Recall:    0.9136
F1:        0.9074
Accuracy:  0.9895 <br>
2e-5, 5 (validation_prediction_5.tsv):
Precision: 0.9053
Recall:    0.9132
F1:        0.9092
Accuracy:  0.9892 <br>
3e-5, 5 (validation_prediction_6.tsv):
Precision: 0.9068
Recall:    0.9175
F1:        0.9122
Accuracy:  0.9893



***

# Question 5

Report some of your best results here. What's your strategy of tuning the hyperparameters?

**Your Answer Here** <br>
I had written a text box above that stored the test results and parameters used. My best result was using the initial setting I had and changing learning rate to 3e-5 and epoch to 5. My strategy of tuning these hyperparameters was to take the standard values I started with and doing a small search in nearby values for learning rate and epoch. Once I had the best of the three values for each of these values, I did one last run with the best individual values to see how it performed, and it performed the best (according to the f1 score).

***

## 1.10 Generate predictions on the test set

After finalizing hyperparameters, run inference on the test set and convert predicted IDs back to label names.

Then write predictions to a TSV file in the required BIO-style format for submission.

***

# TODO 7

In [ ]:
#################################################################
# TODO: Run prediction on the test split.
# Convert logits to predicted IDs.
# Hint: Refer to the code above in the validation set
# (~2 lines)
#predictions, labels, metrics = "YOUR ANSWER HERE"
#predicted_ids = "YOUR ANSWER HERE"
predictions, labels, metrics = trainer.predict(tokenized_datasets["test"])
predicted_ids = np.argmax(predictions, axis=-1)
#################################################################

test_pred_labels = []

test_pred_labels = []

#################################################################
# TODO: Filter out ignored tokens and decode predicted IDs to string labels.
# 1. Iterate through `predicted_ids` and `labels` simultaneously.
# 2. For each sentence, initialize an empty list to store valid predictions.
# 3. Iterate through each token's predicted ID and true label ID.
# 4. If the true label ID indicates the token should be ignored (i.e., -100), skip it.
# 5. Otherwise, convert the predicted ID back to a string using `id2label` and save it.
# 6. Append the sentence's decoded labels to `test_pred_labels`.
# Hint: You will need nested loops and the `zip()` function.
# Hint: Refer to the code from the validation set above
# (~7 lines)

# YOUR CODE HERE
for prediction, label in zip(predicted_ids, labels):
    valid_predictions = []
    for pred_id, label_id in zip(prediction, label):
        if label_id == -100:
            continue
        valid_predictions.append(id2label[pred_id])
    test_pred_labels.append(valid_predictions)

#################################################################

print(test_sentences[0])
print(test_pred_labels[0])

['Torsade', 'de', 'pointes', 'ventricular', 'tachycardia', 'during', 'low', 'dose', 'intermittent', 'dobutamine', 'treatment', 'in', 'a', 'patient', 'with', 'dilated', 'cardiomyopathy', 'and', 'congestive', 'heart', 'failure', '.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


***

In [ ]:
# Save the results
save_predictions(
    "test_predictions.tsv",
    test_sentences,
    test_pred_labels
)

`test_predictions.tsv` and the notebook are your final submission file. Please make sure the token order and sentence boundaries are unchanged from the original test file. You can run the evaluation code below to assess the model’s performance on the test set, report the final test results, and conduct error analysis.

In [ ]:
!python ner_eval.py --gold datasets/BERT_format/test.tsv --pred test_predictions.tsv

Evaluation Results
------------------
Precision: 0.8849
Recall:    0.9084
F1:        0.8965
Accuracy:  0.9884


# Check

Before you submit, check carefully if you have answered all questions:
- [Question 1](#scrollTo=A7b-dlM5h0QT)
- [Question 2](#scrollTo=3VqCpmD4hUSB)
- [Question 3](#scrollTo=KuIWsVPW8aln)
- [Question 4](#scrollTo=-Y2ko7X-_Ycn)
- [Question 5](#scrollTo=7IXtV_ueDiFM)
- [TODO 1](#scrollTo=QyyY8yyGUWW_)
- [TODO 2](#scrollTo=paVEtN3eUWUp)
- [TODO 3](#scrollTo=Foqn7nYPArGH)
- [TODO 4](#scrollTo=PX5a0lHLUWM7)
- [TODO 5](#scrollTo=VkcqQtLPUWH1)
- [TODO 6](#scrollTo=ouE37lrpRPOx)

In [ ]:
# Saving test_predictions.tsv to google drive
from google.colab import drive
drive.mount('/content/drive')
save_predictions(
    "/content/drive/MyDrive/test_predictions.tsv",
    test_sentences,
    test_pred_labels
)

Mounted at /content/drive


In [75]:
# Checking mismatches/errors
from collections import Counter

all_results = Counter()
for true_seq, pred_seq in zip(dev_labels, val_pred_labels):
    for true, pred in zip(true_seq, pred_seq):
        all_results[(true, pred)] += 1

for (true, pred), count in all_results.most_common():
    print(f"True={true}, Pred={pred}: {count}")

True=O, Pred=O: 109750
True=B, Pred=B: 4964
True=I, Pred=I: 1481
True=O, Pred=B: 372
True=B, Pred=O: 330
True=I, Pred=O: 245
True=O, Pred=I: 236
True=B, Pred=I: 53
True=I, Pred=B: 22
